In [55]:
pip install pandas numpy requests

Note: you may need to restart the kernel to use updated packages.


In [56]:
#pip install spacy transformers torch
!pip install "spacy[transformers]"

In [57]:
pip install neo4j py2neo

Note: you may need to restart the kernel to use updated packages.


In [58]:
pip install scikit-learn networkx

Note: you may need to restart the kernel to use updated packages.


In [59]:
pip install beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.


In [60]:
!python -m spacy download en_core_web_trf

     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/457.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/4

ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\sakun\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\sakun\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\sakun\anaconda3\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\sakun\anaconda3\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\sakun\anaconda3\Lib\http\client.py", line 479, in read
    s = self.fp.read(amt)
        ^^^^^^^^^^^^^^^^^
  File "C:\Users\sakun\anaconda3\Lib\socket.py", line 708, in readinto
    return self._sock.recv_into(b)
           ^

     - ----------------------------------- 12.6/457.4 MB 204.7 kB/s eta 0:36:13
     - ----------------------------------- 12.6/457.4 MB 204.8 kB/s eta 0:36:12
     - ----------------------------------- 12.6/457.4 MB 204.8 kB/s eta 0:36:12
     - ----------------------------------- 12.7/457.4 MB 204.8 kB/s eta 0:36:12
     - ----------------------------------- 12.7/457.4 MB 205.4 kB/s eta 0:36:06
     - ----------------------------------- 12.7/457.4 MB 204.8 kB/s eta 0:36:12
     - ----------------------------------- 12.7/457.4 MB 204.8 kB/s eta 0:36:12
     - ----------------------------------- 12.7/457.4 MB 205.3 kB/s eta 0:36:06
     - ----------------------------------- 12.8/457.4 MB 206.1 kB/s eta 0:35:58
     - ----------------------------------- 12.8/457.4 MB 206.5 kB/s eta 0:35:54
     - ----------------------------------- 12.8/457.4 MB 207.0 kB/s eta 0:35:49
     - ----------------------------------- 12.8/457.4 MB 206.5 kB/s eta 0:35:53
     - ---------------------------------

In [44]:
import pandas as pd
import numpy as np
import requests
import json
import re
import os
from datetime import datetime
from typing import List, Dict, Tuple, Optional
import logging
from pathlib import Path

In [17]:
import spacy
from spacy.matcher import Matcher, PhraseMatcher
from transformers import (
    AutoTokenizer, AutoModel, 
    pipeline, BertTokenizer, BertModel
)
import torch

# Neo4j
from neo4j import GraphDatabase
from py2neo import Graph, Node, Relationship

# ML Libraries
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

In [18]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


In [19]:
def fetch_all_fda_recalls(api_key=None, max_records=None):
    """
    Fetch ALL FDA recall data using pagination
    
    Args:
        api_key: Optional API key for higher rate limits
        max_records: Maximum records to fetch (None = fetch all available)
        
    Returns:
        DataFrame with all recall data
    """
    print(f"\n{'='*60}")
    print("FETCHING ALL FDA RECALLS")
    print(f"{'='*60}")
    
    base_url = "https://api.fda.gov/food/enforcement.json"
    all_data = []
    skip = 0
    batch_size = 1000  # API limit per request
    
    # First request to get total count
    params = {
        "limit": 1,
        "skip": 0
    }
    if api_key:
        params["api_key"] = api_key
    
    try:
        print("Getting total record count...")
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        # Get total available records
        total_available = data['meta']['results']['total']
        print(f"✓ Total records available in FDA database: {total_available:,}")
        
        # Determine how many to fetch
        if max_records is None:
            records_to_fetch = total_available
        else:
            records_to_fetch = min(max_records, total_available)
        
        print(f"Records to fetch: {records_to_fetch:,}")
        print(f"Number of batches needed: {(records_to_fetch + batch_size - 1) // batch_size}")
        
    except Exception as e:
        print(f"✗ Error getting total count: {e}")
        return pd.DataFrame()
    
    # Fetch all batches
    batch_num = 0
    while skip < records_to_fetch:
        batch_num += 1
        current_limit = min(batch_size, records_to_fetch - skip)
        
        print(f"\n--- Batch {batch_num} ---")
        print(f"  Fetching records {skip:,} to {skip + current_limit:,}")
        
        params = {
            "limit": current_limit,
            "skip": skip
        }
        if api_key:
            params["api_key"] = api_key
        
        try:
            response = requests.get(base_url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
            
            if 'results' in data and len(data['results']) > 0:
                batch_df = pd.DataFrame(data['results'])
                all_data.append(batch_df)
                
                print(f"  ✓ Batch {batch_num}: {len(batch_df)} records fetched")
                print(f"  Progress: {len(pd.concat(all_data)):,}/{records_to_fetch:,} "
                      f"({len(pd.concat(all_data))/records_to_fetch*100:.1f}%)")
                
                skip += current_limit
            else:
                print(f"  ✗ No more data available")
                break
                
        except Exception as e:
            print(f"  ✗ Error in batch {batch_num}: {e}")
            print(f"  Continuing with next batch...")
            skip += current_limit
            continue
    
    # Combine all batches
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        
        print(f"\n{'='*60}")
        print(f"FETCH COMPLETE")
        print(f"{'='*60}")
        print(f"Total records fetched: {len(combined_df):,}")
        print(f"DataFrame shape: {combined_df.shape}")
        print(f"Memory usage: {combined_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        
        return combined_df
    else:
        print(f"\n✗ No data fetched")
        return pd.DataFrame()

In [20]:
def fetch_all_fda_recalls_with_progress(api_key=None, max_records=None):
    """
    Fetch ALL FDA recalls with a progress bar
    
    Requires: pip install tqdm
    """
    from tqdm import tqdm
    
    print(f"\n{'='*60}")
    print("FETCHING ALL FDA RECALLS (WITH PROGRESS BAR)")
    print(f"{'='*60}")
    
    base_url = "https://api.fda.gov/food/enforcement.json"
    all_data = []
    batch_size = 1000
    
    # Get total count
    params = {"limit": 1, "skip": 0}
    if api_key:
        params["api_key"] = api_key
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        total_available = data['meta']['results']['total']
        
        records_to_fetch = min(max_records, total_available) if max_records else total_available
        
        print(f"Total available: {total_available:,}")
        print(f"Will fetch: {records_to_fetch:,}")
        
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()
    
    # Fetch with progress bar
    num_batches = (records_to_fetch + batch_size - 1) // batch_size
    
    with tqdm(total=records_to_fetch, desc="Fetching FDA data", unit="records") as pbar:
        for i in range(num_batches):
            skip = i * batch_size
            current_limit = min(batch_size, records_to_fetch - skip)
            
            params = {"limit": current_limit, "skip": skip}
            if api_key:
                params["api_key"] = api_key
            
            try:
                response = requests.get(base_url, params=params, timeout=30)
                response.raise_for_status()
                data = response.json()
                
                if 'results' in data and data['results']:
                    batch_df = pd.DataFrame(data['results'])
                    all_data.append(batch_df)
                    pbar.update(len(batch_df))
                else:
                    break
                    
            except Exception as e:
                print(f"\nError in batch {i+1}: {e}")
                continue
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        print(f"\n✓ Fetched {len(combined_df):,} records")
        return combined_df
    
    return pd.DataFrame()

In [21]:
print("Example 1: Fetch ALL FDA recalls")
df_all = fetch_all_fda_recalls(api_key=None)  # Will take time!
print(f"Total records: {len(df_all)}")

# Example 2: Fetch specific number of records
print("\nExample 2: Fetch 5000 records")
df_5k = fetch_all_fda_recalls(api_key=None, max_records=25000)
print(f"Total records: {len(df_5k)}")

# Example 3: With progress bar (prettier output)
print("\nExample 3: Fetch with progress bar")
df_progress = fetch_all_fda_recalls_with_progress(api_key=None, max_records=25000)

# Example 4: Save to CSV to avoid re-fetching
print("\nExample 4: Save to CSV")
df_all.to_csv('fda_all_recalls.csv', index=False)
print("✓ Saved to fda_all_recalls.csv")

# Example 5: Load from saved CSV
print("\nExample 5: Load from saved CSV")
df_loaded = pd.read_csv('fda_all_recalls.csv')
print(f"Loaded {len(df_loaded)} records from file")



Example 1: Fetch ALL FDA recalls

FETCHING ALL FDA RECALLS
Getting total record count...
✓ Total records available in FDA database: 28,144
Records to fetch: 28,144
Number of batches needed: 29

--- Batch 1 ---
  Fetching records 0 to 1,000
  ✓ Batch 1: 1000 records fetched
  Progress: 1,000/28,144 (3.6%)

--- Batch 2 ---
  Fetching records 1,000 to 2,000
  ✓ Batch 2: 1000 records fetched
  Progress: 2,000/28,144 (7.1%)

--- Batch 3 ---
  Fetching records 2,000 to 3,000
  ✓ Batch 3: 1000 records fetched
  Progress: 3,000/28,144 (10.7%)

--- Batch 4 ---
  Fetching records 3,000 to 4,000
  ✓ Batch 4: 1000 records fetched
  Progress: 4,000/28,144 (14.2%)

--- Batch 5 ---
  Fetching records 4,000 to 5,000
  ✓ Batch 5: 1000 records fetched
  Progress: 5,000/28,144 (17.8%)

--- Batch 6 ---
  Fetching records 5,000 to 6,000
  ✓ Batch 6: 1000 records fetched
  Progress: 6,000/28,144 (21.3%)

--- Batch 7 ---
  Fetching records 6,000 to 7,000
  ✓ Batch 7: 1000 records fetched
  Progress: 7,000/28

Fetching FDA data: 100%|███████████████████████████████████████████████████| 25000/25000 [01:09<00:00, 357.67records/s]



✓ Fetched 25,000 records

Example 4: Save to CSV
✓ Saved to fda_all_recalls.csv

Example 5: Load from saved CSV
Loaded 26000 records from file


In [22]:
def save_fda_data(df, filename='fda_all_recalls.csv'):
    """Save FDA data to CSV"""
    print(f"\nSaving data to {filename}...")
    df.to_csv(filename, index=False)
    print(f"✓ Saved {len(df):,} records")
    print(f"  File size: {os.path.getsize(filename) / 1024**2:.2f} MB")


def load_fda_data(filename='fda_all_recalls.csv'):
    """Load FDA data from CSV"""
    print(f"\nLoading data from {filename}...")
    
    if not os.path.exists(filename):
        print(f"✗ File not found: {filename}")
        return pd.DataFrame()
    
    df = pd.read_csv(filename)
    print(f"✓ Loaded {len(df):,} records")
    print(f"  Columns: {len(df.columns)}")
    print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    return df



In [23]:
def recommended_workflow():
    """
    Recommended workflow: Fetch once, save, then load
    """
    
    # Step 1: Fetch all data (do this ONCE)
    print("STEP 1: Fetching all FDA data (this may take 10-30 minutes)...")
    df_all = fetch_all_fda_recalls(api_key=None)
    
    # Step 2: Save immediately
    print("\nSTEP 2: Saving data...")
    save_fda_data(df_all, 'fda_all_recalls.csv')
    
    # Step 3: For future runs, just load from CSV
    print("\nSTEP 3: Loading from saved file...")
    df = load_fda_data('fda_all_recalls.csv')
    
    # Step 4: Continue with preprocessing
    print("\nSTEP 4: Preprocessing...")
    df_clean = preprocess_fda_data(df)
    
    return df_clean

In [36]:
def handle_missing_values(df):
    """
    Handle missing values in DataFrame
    
    Args:
        df: DataFrame
        
    Returns:
        DataFrame with handled missing values
    """
    print(f"\n{'='*60}")
    print("HANDLING MISSING VALUES")
    print(f"{'='*60}")
    
    # Count missing values before
    print("Missing values before:")
    for col in df.columns:
        missing = df[col].isna().sum()
        if missing > 0:
            print(f"  {col}: {missing} ({missing/len(df)*100:.1f}%)")
    
    # Fill text columns
    text_columns = ['reason', 'product', 'company']
    for col in text_columns:
        if col in df.columns:
            df[col] = df[col].fillna('')
            print(f"✓ Filled '{col}' with empty string")
    
    # Fill categorical columns
    if 'classification' in df.columns:
        df['classification'] = df['classification'].fillna('Unknown')
        print(f"✓ Filled 'classification' with 'Unknown'")
    
    if 'status' in df.columns:
        df['status'] = df['status'].fillna('Unknown')
        print(f"✓ Filled 'status' with 'Unknown'")
    
    print(f"✓ Missing values handled")
    return df



In [37]:
def clean_text_fields(df):
    """
    Clean text fields by removing extra whitespace and special characters
    
    Args:
        df: DataFrame
        
    Returns:
        Cleaned DataFrame
    """
    print(f"\n{'='*60}")
    print("CLEANING TEXT FIELDS")
    print(f"{'='*60}")
    
    text_columns = ['reason', 'product', 'company', 'distribution']
    
    for col in text_columns:
        if col in df.columns:
            print(f"Cleaning '{col}'...")
            
            # Show before
            if len(df[col]) > 0 and df[col].iloc[0]:
                print(f"  Before: {df[col].iloc[0][:80]}...")
            
            # Remove extra whitespace
            df[col] = df[col].str.replace(r'\s+', ' ', regex=True)
            
            # Remove special characters
            df[col] = df[col].str.replace(r'[^\w\s,.\-()%/]', '', regex=True)
            
            # Strip whitespace
            df[col] = df[col].str.strip()
            
            # Show after
            if len(df[col]) > 0 and df[col].iloc[0]:
                print(f"  After: {df[col].iloc[0][:80]}...")
            
            print(f"  ✓ '{col}' cleaned")
    
    print(f"✓ All text fields cleaned")
    return df


In [38]:
def select_and_rename_columns(df):
    """Select relevant columns and rename them"""
    
    # PRINT 1: Show original columns
    print(f"\n{'='*60}")
    print("COLUMN SELECTION - BEFORE")
    print(f"{'='*60}")
    print(f"Original columns ({len(df.columns)}):")
    for col in df.columns:
        print(f" - {col}")
   
    column_mapping = {
        'recall_number': 'recall_id',
        'reason_for_recall': 'reason',
        'product_description': 'product',
        'classification': 'classification',
        'status': 'status',
        'distribution_pattern': 'distribution',
        'recall_initiation_date': 'recall_date',
        'recalling_firm': 'company',
        'city': 'city',
        'state': 'state',
        'country': 'country',
        'product_quantity': 'quantity',
        'code_info': 'code_info'
    }
   
    # Select only existing columns from the mapping
    existing_columns = [col for col in column_mapping.keys() if col in df.columns]
    missing_columns = [col for col in column_mapping.keys() if col not in df.columns]
   
    if missing_columns:
        print(f"\n{'='*60}")
        print("MISSING COLUMNS")
        print(f"{'='*60}")
        print("The following mapped columns are missing from the DataFrame and will be skipped:")
        for col in missing_columns:
            print(f" - {col}")
   
    existing_columns = {old: new for old, new in column_mapping.items() if old in df.columns}
    missing_columns = [old for old in column_mapping.keys() if old not in df.columns]
   
    if missing_columns:
        print(f"\nWarning: These expected columns are MISSING and will be skipped:")
        for col in missing_columns:
            print(f" - {col}")
   
    # Keep only existing wanted columns
    df = df[list(existing_columns.keys())]
   
    # Rename them
    df = df.rename(columns=existing_columns)

    # PRINT 2: Show columns after selection and renaming
    print(f"\n{'='*60}")
    print("COLUMN SELECTION - AFTER")
    print(f"{'='*60}")
    print(f"Selected and renamed columns ({len(df.columns)}):")
    for col in df.columns:
        print(f" - {col}")

   
    return df


In [39]:
def parse_dates(df):
    """
    Parse date fields
    
    Args:
        df: DataFrame
        
    Returns:
        DataFrame with parsed dates
    """
    print(f"\n{'='*60}")
    print("PARSING DATES")
    print(f"{'='*60}")
    
    if 'recall_date' not in df.columns:
        print("✗ No 'recall_date' column found")
        return df
    
    print(f"Original date format: {df['recall_date'].iloc[0] if len(df) > 0 else 'N/A'}")
    
    df['recall_date'] = pd.to_datetime(
        df['recall_date'],
        format='%Y%m%d',
        errors='coerce'
    )
    
    print(f"Parsed date format: {df['recall_date'].iloc[0] if len(df) > 0 else 'N/A'}")
    
    # Extract year and month
    df['recall_year'] = df['recall_date'].dt.year
    df['recall_month'] = df['recall_date'].dt.month
    
    print(f"✓ Dates parsed successfully")
    print(f"  Date range: {df['recall_date'].min()} to {df['recall_date'].max()}")
    
    return df


In [40]:
def extract_classification(df):
    """
    Extract classification and severity levels
    
    Args:
        df: DataFrame
        
    Returns:
        DataFrame with severity levels
    """
    print(f"\n{'='*60}")
    print("EXTRACTING CLASSIFICATION INFO")
    print(f"{'='*60}")
    
    if 'classification' not in df.columns:
        print("✗ No 'classification' column found")
        return df
    
    # Show classification distribution
    print("Classification distribution:")
    print(df['classification'].value_counts())
    
    # Map to severity
    classification_map = {
        'Class I': 'HIGH',
        'Class II': 'MEDIUM',
        'Class III': 'LOW'
    }
    
    df['severity'] = df['classification'].map(
        lambda x: classification_map.get(x, 'UNKNOWN')
    )
    
    print("\nSeverity distribution:")
    print(df['severity'].value_counts())
    
    # Extract class number
    df['class_number'] = df['classification'].str.extract(
        r'Class\s+([I]+)', expand=False
    )
    
    print(f"✓ Classification extracted")
    return df


In [41]:
def create_unique_ids(df):
    """
    Create unique identifiers for recalls
    
    Args:
        df: DataFrame
        
    Returns:
        DataFrame with unique IDs
    """
    print(f"\n{'='*60}")
    print("CREATING UNIQUE IDs")
    print(f"{'='*60}")
    
    if 'recall_id' not in df.columns:
        df['recall_id'] = [f"FDA_RECALL_{i:05d}" for i in range(len(df))]
        print(f"✓ Created new recall IDs")
        print(f"  Sample: {df['recall_id'].iloc[0]}")
    else:
        # Clean existing IDs
        df['recall_id'] = df['recall_id'].str.replace(r'[^\w\-]', '_', regex=True)
        print(f"✓ Cleaned existing recall IDs")
        print(f"  Sample: {df['recall_id'].iloc[0]}")
    
    print(f"  Total unique IDs: {df['recall_id'].nunique()}")
    return df



In [42]:
def preprocess_fda_data(df):
    """Complete preprocessing pipeline"""
    
    print(f"\n{'#'*60}")
    print(f"# STARTING PREPROCESSING PIPELINE")
    print(f"{'#'*60}")
    print(f"Initial shape: {df.shape}")
    
    if df.empty:
        print("✗ Empty DataFrame provided")
        return df
    
    # Step 1: Select columns
    df = select_and_rename_columns(df)
    
    # Step 2: Handle missing values
    df = handle_missing_values(df)
    
    # Step 3: Clean text
    df = clean_text_fields(df)
    
    # Step 4: Parse dates
    df = parse_dates(df)
    
    # Step 5: Extract classification
    df = extract_classification(df)
    
    # Step 6: Create IDs
    df = create_unique_ids(df)

    
    
    print(f"\n{'#'*60}")
    print(f"# PREPROCESSING COMPLETE")
    print(f"{'#'*60}")
    print(f"Final shape: {df.shape}")
    print(f"Final columns: {list(df.columns)}")
    
    save_fda_data(df, filename='fda_preprocessed.csv')
    
    
    return df

In [45]:
if __name__ == "__main__":
    
    print("="*70)
    print("FDA DATA PROCESSING PIPELINE - STARTING")
    print("="*70)
    print("\n[STEP 1] Fetching FDA data...")
    df_raw = fetch_all_fda_recalls(
        api_key=None,           # Add your API key here if you have one
        max_records=None        # Start with 1000 for testing, then use None for all
    )
    
    # Check if data was fetched
    if df_raw.empty:
        print("❌ No data fetched! Exiting...")
        exit()
    
    # Save raw data
    print("\nSaving raw data...")
    df_raw.to_csv('data/fda_raw.csv', index=False)
    print(f"✓ Saved to data/fda_raw.csv")

    print("\n[STEP 2] Preprocessing data...")
    df_clean = preprocess_fda_data(df_raw)
    
    # Save preprocessed data
    df_clean.to_csv('data/fda_preprocessed.csv', index=False)
    print(f"✓ Saved to data/fda_preprocessed.csv")

  

    print("\n" + "="*70)
    print("PIPELINE COMPLETE!")
    print("="*70)
    print(f"📊 Summary:")
    print(f"  - Raw records fetched: {len(df_raw):,}")
    print(f"  - Preprocessed records: {len(df_clean):,}")
    print("="*70)




FDA DATA PROCESSING PIPELINE - STARTING

[STEP 1] Fetching FDA data...

FETCHING ALL FDA RECALLS
Getting total record count...
✓ Total records available in FDA database: 28,144
Records to fetch: 28,144
Number of batches needed: 29

--- Batch 1 ---
  Fetching records 0 to 1,000
  ✓ Batch 1: 1000 records fetched
  Progress: 1,000/28,144 (3.6%)

--- Batch 2 ---
  Fetching records 1,000 to 2,000
  ✓ Batch 2: 1000 records fetched
  Progress: 2,000/28,144 (7.1%)

--- Batch 3 ---
  Fetching records 2,000 to 3,000
  ✓ Batch 3: 1000 records fetched
  Progress: 3,000/28,144 (10.7%)

--- Batch 4 ---
  Fetching records 3,000 to 4,000
  ✓ Batch 4: 1000 records fetched
  Progress: 4,000/28,144 (14.2%)

--- Batch 5 ---
  Fetching records 4,000 to 5,000
  ✓ Batch 5: 1000 records fetched
  Progress: 5,000/28,144 (17.8%)

--- Batch 6 ---
  Fetching records 5,000 to 6,000
  ✓ Batch 6: 1000 records fetched
  Progress: 6,000/28,144 (21.3%)

--- Batch 7 ---
  Fetching records 6,000 to 7,000
  ✓ Batch 7: 100